# 08 — Handling Missing & Null Data

**What you will learn:**
- Detecting nulls: `isNull()`, `isNotNull()`, counting nulls per column
- `dropna()` — drop rows with nulls (with `how`, `thresh`, `subset`)
- `fillna()` / `fill()` — fill nulls with static values
- `replace()` — replace specific values
- `coalesce()` — first non-null from multiple columns
- `na.drop()`, `na.fill()`, `na.replace()` — the DataFrameNaFunctions API
- Handling NaN (Not-a-Number) vs NULL
- Imputation patterns (mean/median imputation)

## Setup — DataFrame with Intentional Nulls

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, isnan, isnull, count, when, coalesce, lit, mean

schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("city",       StringType(),  True),
    StructField("bonus_pct",  DoubleType(),  True),
])

data = [
    (1,  "Alice",   "Engineering", 95000.0,  "NYC",     15.0),
    (2,  "Bob",     "Marketing",   72000.0,  None,      10.0),
    (3,  "Charlie", "Engineering", None,     "SF",      None),
    (4,  "Diana",   None,          68000.0,  "NYC",     8.0),
    (5,  "Eve",     "Marketing",   78000.0,  "Chicago", None),
    (6,  None,      "HR",          71000.0,  None,      12.0),
    (7,  "Grace",   "HR",          None,     None,      None),
    (8,  "Hank",    "Engineering", 92000.0,  "SF",      18.0),
]

df = spark.createDataFrame(data, schema=schema)
df.show()

## 1. Detecting Nulls

In [ ]:
# Filter rows where salary is null
df.filter(col("salary").isNull()).show()

# Filter rows where city is NOT null
df.filter(col("city").isNotNull()).show()

In [ ]:
# Count nulls in each column
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show()

In [ ]:
# Percentage of nulls per column
row_count = df.count()
null_pct = df.select([
    (count(when(col(c).isNull(), c)) / row_count * 100).alias(c)
    for c in df.columns
])
null_pct.show()

## 2. NaN vs NULL

| | NULL | NaN |
|---|---|---|
| Meaning | Missing / unknown value | "Not a Number" (result of invalid math) |
| Type | Any type | Float / Double only |
| Detection | `isNull()` | `isnan()` |
| In agg | Ignored by sum/avg | Propagates (makes result NaN) |

In [ ]:
import math
from pyspark.sql.functions import isnan

nan_data = [(1, 95000.0), (2, float("nan")), (3, None), (4, 80000.0)]
df_nan = spark.createDataFrame(nan_data, ["id", "salary"])

df_nan.select(
    col("id"),
    col("salary"),
    isnull(col("salary")).alias("is_null"),
    isnan(col("salary")).alias("is_nan"),
    (isnull(col("salary")) | isnan(col("salary"))).alias("is_null_or_nan")
).show()

## 3. dropna() — Drop Rows with Nulls

In [ ]:
# Drop rows where ANY column is null
df.dropna(how="any").show()

In [ ]:
# Drop rows where ALL columns are null
df.dropna(how="all").show()

In [ ]:
# Drop rows where specific columns have nulls
df.dropna(subset=["salary", "department"]).show()

In [ ]:
# Drop rows unless at least N non-null columns (thresh)
# Keep rows with at least 5 non-null values out of 6 columns
df.dropna(thresh=5).show()

## 4. fillna() / fill() — Fill Nulls with Static Values

In [ ]:
# Fill all string nulls and all numeric nulls with separate values
df.fillna("UNKNOWN").show()                  # fills only string columns

In [ ]:
df.fillna(0.0).show()                        # fills only numeric columns

In [ ]:
# Fill specific columns with specific values using a dict
df.fillna({
    "department": "Unknown",
    "city":       "Remote",
    "salary":     0.0,
    "bonus_pct":  0.0
}).show()

In [ ]:
# Fill per column using withColumn + coalesce (more expressive)
df_filled = (
    df
    .withColumn("department", coalesce(col("department"), lit("Unknown")))
    .withColumn("city",       coalesce(col("city"),       lit("Remote")))
    .withColumn("name",       coalesce(col("name"),       lit("N/A")))
)
df_filled.show()

## 5. Mean / Median Imputation for Numeric Columns

In [ ]:
# Calculate mean salary from non-null rows
mean_salary = df.select(mean("salary")).collect()[0][0]
print(f"Mean salary: {mean_salary:.2f}")

mean_bonus  = df.select(mean("bonus_pct")).collect()[0][0]
print(f"Mean bonus:  {mean_bonus:.2f}")

# Fill nulls with the computed mean
df_imputed = df.fillna({
    "salary":    round(mean_salary, 2),
    "bonus_pct": round(mean_bonus, 2)
})
df_imputed.show()

## 6. replace() — Replace Specific Values (including nulls → value)

In [ ]:
# Replace a specific value in a column
df.na.replace("Marketing", "Digital Marketing", subset=["department"]).show()

In [ ]:
# Replace multiple values
df.na.replace(
    {"NYC": "New York City", "SF": "San Francisco"},
    subset=["city"]
).show()

## 7. coalesce() — First Non-Null from Multiple Columns

In [ ]:
# Pick city; if null, use department; if still null, use "Unknown"
df.select(
    col("emp_id"),
    col("name"),
    col("city"),
    col("department"),
    coalesce(col("city"), col("department"), lit("Unknown")).alias("location_fallback")
).show()

## 8. Forward Fill (Last Value Carried Forward)

Spark doesn't have a native forward-fill, but we can simulate it using Window + last() with `ignorenulls=True`.

In [ ]:
from pyspark.sql.functions import last
from pyspark.sql.window import Window

# Time-series data with missing values
ts_data = [
    ("2024-01", 100.0),
    ("2024-02", None),
    ("2024-03", None),
    ("2024-04", 120.0),
    ("2024-05", None),
    ("2024-06", 130.0),
]
df_ts = spark.createDataFrame(ts_data, ["month", "value"])

window_ffill = (
    Window.orderBy("month")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_ffill = df_ts.withColumn(
    "value_ffill",
    last(col("value"), ignorenulls=True).over(window_ffill)
)
df_ffill.show()

## 9. Null-Safe Equality Check

Normal `==` returns null when comparing null to anything.
Use `eqNullSafe` (`<=>` in SQL) when you want null == null to be True.

In [ ]:
df1 = spark.createDataFrame([(1, "NYC"), (2, None), (3, "SF")], ["id", "city"])
df2 = spark.createDataFrame([(1, "NYC"), (2, None), (3, "LA")], ["id", "city"])

# Normal join — null != null, so row 2 doesn't match
df1.join(df2, df1["city"] == df2["city"], "inner").show()

In [ ]:
# Null-safe join — null == null is True
df1.join(df2, df1["city"].eqNullSafe(df2["city"]), "inner").show()

## Key Takeaways

| Task | Method |
|---|---|
| Detect nulls | `col.isNull()` / `col.isNotNull()` |
| Detect NaN | `isnan(col)` |
| Count nulls per column | `count(when(col.isNull(), col))` |
| Drop null rows | `df.dropna(how, thresh, subset)` |
| Fill with static value | `df.fillna(value)` or `df.fillna({col: val})` |
| Fill with mean | Compute mean → `fillna(mean_value)` |
| First non-null | `coalesce(col1, col2, lit("default"))` |
| Replace values | `df.na.replace(old, new, subset)` |
| Forward fill | `last(col, ignorenulls=True).over(window)` |
| Null-safe equality | `col1.eqNullSafe(col2)` |